In [ ]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/ARR Data/Human Validation/humanData.xlsx" .

In [ ]:
!pip install hazm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9371 sha256=e77bddea5ca99aa356ccae4ef79c6f026be69ce81f0a6e2c5e8ab613d315a137
  Stored in directory: /root/.cache/pip/wheels/be/12/c9/228313ff5cb722777830302f1d4136de4129932e6a0354c056
Successfully built flashtext


In [ ]:
!pip install -q "transformers<5" "tokenizers<0.22"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import numpy as np
import pandas as pd
import ast
from hazm import WordTokenizer
import torch

In [ ]:
df = pd.read_excel("humanData.xlsx")

In [ ]:
annotator1_col = 'Hop_5_Rationale_Tactic_Span (A1)'
annotator2_col = 'Hop_5_Rationale_Tactic_Span (A2)'

In [ ]:
df = df[df["Hop_1_Triage (A1)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage (A2)"] != "بی‌ضرر"]
print(len(df))

68


In [ ]:
import ast

for idx, value in df[annotator2_col].items():
    try:
        ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError) as e:
        print(f"Index: {idx}, Value: {value!r}, Error: {e}")

In [ ]:
df[annotator1_col] = df[annotator1_col].apply(ast.literal_eval)
df[annotator2_col] = df[annotator2_col].apply(ast.literal_eval)

In [ ]:
tokenizer = WordTokenizer()

def tokenize(text):
    return tokenizer.tokenize(text)

In [ ]:
def jaccard(a, b):
    A, B = set(tokenize(a)), set(tokenize(b))
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

In [ ]:
def normal_jaccard(pred_spans, gold_spans):
    if not pred_spans or not gold_spans:
        return 0.0

    scores = []
    for p in pred_spans:
        scores.append(max(jaccard(p, g) for g in gold_spans))
    return sum(scores) / len(scores)

In [ ]:
def evaluate_rationales(df, annotator1_col, annotator2_col):
    jaccards = []

    for index, row in df.iterrows():
        jaccard_val = normal_jaccard(row[annotator2_col], row[annotator1_col])
        jaccards.append(jaccard_val)

    return {
        "normalJaccard": sum(jaccards) / len(jaccards)
    }

In [ ]:
results = evaluate_rationales(df, annotator1_col, annotator2_col)

In [ ]:
results

{'normalJaccard': 0.80265522875817}